From this video:
https://www.youtube.com/watch?v=aFnHYEv71U4

In tensorflow, but useful concepts

In [1]:
import numpy as np
import spektral
import tensorflow as tf

In [3]:
dataset = spektral.datasets.citation.Cora()
graph = dataset[0]

features = graph.x
adjacency = graph.a
labels = graph.y

adj = adjacency.toarray()  # convert to dense numpy
adj = tf.convert_to_tensor(adj, dtype=tf.float32)

print(features.shape) #features
print(adj.shape) #adjacency
print(labels.shape) #labels

print(np.sum(dataset.mask_tr))
print(np.sum(dataset.mask_va))
print(np.sum(dataset.mask_te))


(2708, 1433)
(2708, 2708)
(2708, 7)
140
500
1000


/Users/jorgemy/anaconda3/envs/torch-arm/lib/python3.10/site-packages/scipy/sparse/_index.py:146: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil_matrix is more efficient.
  self._set_arrayXarray(i, j, x)


In [4]:
print(features)

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [7]:
print(adj)

tf.Tensor(
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 1. ... 0. 0. 0.]
 [0. 1. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 1.]
 [0. 0. 0. ... 0. 1. 0.]], shape=(2708, 2708), dtype=float32)


In [6]:
print(labels)

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 1. 0. 0.]
 [0. 0. 0. ... 1. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [16]:
def masked_softmax_cross_entropy(logits, labels, mask):
    loss = tf.nn.softmax_cross_entropy_with_logits(logits=logits, labels=labels)
    mask = tf.cast(mask, dtype=tf.float32)
    mask /= tf.reduce_mean(mask)
    loss *= mask
    return tf.reduce_mean(loss)

def masked_accuracy(logits, labels, mask):
    correct_prediction = tf.equal(tf.argmax(logits, 1), tf.argmax(labels, 1))
    accuracy_all = tf.cast(correct_prediction,tf.float32)
    mask = tf.cast(mask, dtype=tf.float32)
    mask /= tf.reduce_mean(mask)
    accuracy_all *= mask
    return tf.reduce_mean(accuracy_all)

In [7]:
def gnn(fts,  adj, transform, activation):
    seq_fits = transform(fts)
    ret_fts = tf.matmul(adj, seq_fits)
    return activation(ret_fts)

In [14]:
def train_cora(fts, adj, gnn_fn, units, epochs, lr):
    lyr_1 = tf.keras.layers.Dense(units)
    lyr_2 = tf.keras.layers.Dense(7)

    def cora_gnn(fts, adj):
        hidden = gnn_fn(fts, adj, lyr_1, tf.nn.relu)
        logits = gnn_fn(hidden, adj, lyr_2, tf.identity)
        return logits
    
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
    best_accuracy = 0.0
    for ep in range(epochs+1):
        with tf.GradientTape() as t:
            logits = cora_gnn(fts, adj)
            loss = masked_softmax_cross_entropy(logits, labels, dataset.mask_tr)
        
        variables = t.watched_variables()
        grads = t.gradient(loss, variables)
        optimizer.apply_gradients(zip(grads, variables))
        logits = cora_gnn(fts, adj)
        val_accuracy = masked_accuracy(logits, labels, dataset.mask_va)
        test_accuracy = masked_accuracy(logits, labels, dataset.mask_te)

        if val_accuracy > best_accuracy:
            best_accuracy = val_accuracy
            print(f'Epoch: {ep}, Loss: {loss.numpy()}, val accuacy: {val_accuracy.numpy()}, test accuracy: {test_accuracy.numpy()}')
            


In [17]:
train_cora(features, adj, gnn, 32, 200, 0.01)

Epoch: 0, Loss: 3.91668701171875, val accuacy: 0.29999998211860657, test accuracy: 0.3310000002384186
Epoch: 2, Loss: 2.71479868888855, val accuacy: 0.38599997758865356, test accuracy: 0.40199998021125793
Epoch: 3, Loss: 1.807702660560608, val accuacy: 0.5400000214576721, test accuracy: 0.5400000214576721
Epoch: 4, Loss: 1.205180287361145, val accuacy: 0.6259999871253967, test accuracy: 0.6429999470710754
Epoch: 5, Loss: 0.9920770525932312, val accuacy: 0.671999990940094, test accuracy: 0.6669999957084656
Epoch: 6, Loss: 0.8324085474014282, val accuacy: 0.6779999732971191, test accuracy: 0.6620000004768372
Epoch: 7, Loss: 0.9174574017524719, val accuacy: 0.6939999461174011, test accuracy: 0.6869999766349792
Epoch: 8, Loss: 0.7041504979133606, val accuacy: 0.731999933719635, test accuracy: 0.718999981880188
Epoch: 9, Loss: 0.5841679573059082, val accuacy: 0.7379999756813049, test accuracy: 0.7229999899864197
Epoch: 19, Loss: 0.2132771909236908, val accuacy: 0.7400000095367432, test accu

In [18]:
deg = tf.reduce_sum(adj, axis=-1) # mean pooling
train_cora(features, adj/deg, gnn, 32, 200, 0.01)

Epoch: 0, Loss: 1.9322091341018677, val accuacy: 0.40799999237060547, test accuracy: 0.41199997067451477
Epoch: 1, Loss: 1.7373169660568237, val accuacy: 0.5360000133514404, test accuracy: 0.5519999861717224
Epoch: 2, Loss: 1.5295385122299194, val accuacy: 0.6080000400543213, test accuracy: 0.6240000128746033
Epoch: 3, Loss: 1.314582347869873, val accuacy: 0.6359999775886536, test accuracy: 0.6499999761581421
Epoch: 4, Loss: 1.114460825920105, val accuacy: 0.6539999842643738, test accuracy: 0.6790000200271606
Epoch: 5, Loss: 0.93287593126297, val accuacy: 0.6859999895095825, test accuracy: 0.7020000219345093
Epoch: 6, Loss: 0.7710557579994202, val accuacy: 0.7140000462532043, test accuracy: 0.7259999513626099
Epoch: 7, Loss: 0.6306949257850647, val accuacy: 0.7399999499320984, test accuracy: 0.7519999146461487
Epoch: 8, Loss: 0.5114647746086121, val accuacy: 0.7479999661445618, test accuracy: 0.7689998745918274
Epoch: 9, Loss: 0.4131001830101013, val accuacy: 0.7620000243186951, test a

### Using Pytorch - torch_gemoetric

In [13]:
import torch
from torch_geometric.data import Data

In [16]:
# [source, target]
edge_index = torch.tensor([[0, 1],
                           [1, 0],
                           [1, 2],
                           [2, 1]], dtype=torch.long)
x = torch.tensor([[-1], [0], [1]], dtype=torch.float)

data = Data(x=x, edge_index=edge_index.t().contiguous())

In [ ]:
# some checks and tests that the Data object provides
print(f"Num nodes: {data.num_nodes}")

print(f"Num edges: {data.num_edges}")

print(f"Num node features: {data.num_node_features}")

print(f"Isolated nodes? {data.has_isolated_nodes()}")

print(f"Self loops? {data.has_self_loops()}")
    
print(f"Is directed? {data.is_directed()}")

Num nodes: 3
Num edges: 4
Num node features: 1
Isolated nodes? False
Self loops? False
Is directed? False
